# 🎓 Flan-T5-Large Content Specialist — Phase 2: Evaluation & Finalization (Kaggle 2×T4)

This notebook **continues** the RTX 5090 training run. Training is already done
(3 epochs → `checkpoint-790 / 1580 / 2370`). Here we:

1. Load the uploaded checkpoints (no retraining).
2. Pick the **best epoch by validation loss** (the original run used
   `eval_strategy="no"`, so no best checkpoint was recorded).
3. Run **OOM-safe batched generation** and compute an expanded metric suite.
4. Produce thesis-grade **visualizations** (saved to session storage).
5. Save the **final model** + model card.

---
## 📦 What to upload to Kaggle (and what to delete first)

Each `checkpoint-*` folder from training contains files only needed to *resume*
training. For evaluation you only need the **weights + tokenizer**. Before
zipping/uploading, **delete these from every checkpoint** (they are huge and
useless here):

- `optimizer.pt`  ← biggest (~2× model size)
- `rng_state.pth`
- `scheduler.pt`

**Keep:** `config.json`, `generation_config.json`, `model.safetensors`,
`tokenizer_config.json`, `tokenizer.json`, `spiece.model` (if present), and
`trainer_state.json` (tiny — used for the inventory).

Then upload as **two Kaggle datasets** (or one combined):
- the `checkpoints/` folder (the 3 trimmed checkpoints)
- `content_train_cleaned.jsonl` (the SAME dataset file used in training)

Both are auto-discovered under `/kaggle/input`.

> **Hardware:** Set accelerator to **GPU T4 ×2**. We only need **one** T4 —
> flan-t5-large (770M) runs in **fp32** (~3 GB), which avoids both the previous
> OOM (caused by `trainer.predict` returning full logits) and T5's fp16 NaN
> instability. The second GPU sits idle; that's fine.

In [ ]:
# 1a — Install (most are preinstalled on Kaggle; pin the metric libs)
!pip -q install -U "evaluate>=0.4" rouge-score bert-score sacrebleu nltk 2>/dev/null
import transformers, torch
print("transformers:", transformers.__version__, "| torch:", torch.__version__,
      "| CUDA:", torch.cuda.is_available())

In [ ]:
# 1b — Imports, GPU detection (T4 → fp32), seeds, and plot theme
import os, sys, json, glob, time, random, re, datetime
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cycler
import seaborn as sns
from tqdm.auto import tqdm
from IPython.display import display

import nltk
for pkg in ["punkt", "punkt_tab", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
    g = torch.cuda.get_device_properties(0)
    print(f"GPU: {g.name} | VRAM: {g.total_memory/1e9:.1f} GB | count: {torch.cuda.device_count()}")
else:
    DEVICE = "cpu"; print("⚠️  No GPU — running on CPU (slow).")

# T5 is unstable in fp16 and T4 lacks bf16 → use fp32 (model is small, fits in 16 GB).
EVAL_DTYPE = torch.float32
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ── Plot aesthetics (saved figures are downloadable from the Output tab) ──
FIG_DIR = Path("/kaggle/working/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3",
           "#937860", "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD"]
ACCENT, GRID = "#2D3142", "#E7E7EE"
sns.set_theme(style="white", context="notebook")
plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 200, "figure.facecolor": "white",
    "savefig.facecolor": "white", "savefig.bbox": "tight",
    "axes.edgecolor": "#B9B9C6", "axes.linewidth": 1.0,
    "axes.titlesize": 14, "axes.titleweight": "bold", "axes.titlepad": 14,
    "axes.titlecolor": ACCENT, "axes.labelcolor": ACCENT, "axes.labelsize": 11,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GRID, "grid.linewidth": 1.0,
    "xtick.color": ACCENT, "ytick.color": ACCENT, "xtick.labelsize": 10, "ytick.labelsize": 10,
    "axes.prop_cycle": cycler(color=PALETTE), "font.size": 11, "legend.frameon": False,
    "figure.titleweight": "bold", "figure.titlesize": 16,
})

def _slug(s): return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")
def _despine(ax, left=True, bottom=True):
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    if not left: ax.spines["left"].set_visible(False)
    if not bottom: ax.spines["bottom"].set_visible(False)
def save_fig(fig, name):
    p = FIG_DIR / f"{name}.png"; fig.savefig(p); print("  💾", p); plt.show(); plt.close(fig)

print("🎨 Theme ready · figures →", FIG_DIR, "· dtype:", EVAL_DTYPE)

In [ ]:
# 1c — Configuration + explicit dataset/checkpoint paths
class CFG:
    base_model      = "google/flan-t5-large"
    test_size       = 0.10          # MUST match training to reproduce the split
    val_size        = 0.10
    max_input_len   = 512
    max_target_len  = 256
    # OOM-safe generation knobs (tune down if you ever see OOM)
    gen_batch_size  = 8
    gen_num_beams   = 4
    loss_batch_size = 8
    # Cap eval samples for a quick pass (None = full split). Full test is best for the thesis.
    eval_max_samples = None
    out_dir         = Path("/kaggle/working/content_specialist_final")

# ── Explicit input paths (Kaggle datasets) ──────────────────────────────────
DATASET_PATH = "/kaggle/input/datasets/seifmahdy1/slides-generation/content_train_cleaned.jsonl"
CKPT_PATHS = [
    Path("/kaggle/input/datasets/seifmahdy1/checkpoint-1/checkpoint-790"),
    Path("/kaggle/input/datasets/seifmahdy1/checkpoint-2/checkpoint-1580"),
    Path("/kaggle/input/datasets/seifmahdy1/checkpoint-3/checkpoint-2370"),
]
# Keep epoch order (790 → 1580 → 2370)
CKPT_PATHS = sorted(CKPT_PATHS, key=lambda p: int(p.name.split("-")[1]))

CFG.out_dir.mkdir(parents=True, exist_ok=True)

# ── Sanity check (fail fast on a wrong path) ────────────────────────────────
assert Path(DATASET_PATH).exists(), f"❌ dataset not found: {DATASET_PATH}"
for ck in CKPT_PATHS:
    assert (ck / "config.json").exists(), f"❌ checkpoint missing config.json: {ck}"
    assert (ck / "model.safetensors").exists() or (ck / "pytorch_model.bin").exists(), \
        f"❌ checkpoint missing weights: {ck}"

print("Dataset:    ", DATASET_PATH)
print("Checkpoints:", [p.name for p in CKPT_PATHS])

In [ ]:
# 2a — Load dataset and reproduce the EXACT train/val/test split (seed=42)
from sklearn.model_selection import train_test_split

raw_data = []
with open(DATASET_PATH) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            o = json.loads(line)
            if o.get("input", "").strip() and o.get("target", "").strip():
                raw_data.append(o)
        except json.JSONDecodeError:
            pass

train_val, test_data = train_test_split(raw_data, test_size=CFG.test_size, random_state=SEED)
train_data, val_data = train_test_split(
    train_val, test_size=CFG.val_size / (1 - CFG.test_size), random_state=SEED)
print(f"Total: {len(raw_data):,}  |  Train: {len(train_data):,}  Val: {len(val_data):,}  Test: {len(test_data):,}")

# Choose the held-out split for headline generation metrics
EVAL_NAME = "test"
eval_data = test_data if EVAL_NAME == "test" else val_data
if CFG.eval_max_samples:
    eval_data = eval_data[:CFG.eval_max_samples]
print(f"Headline metrics on: {EVAL_NAME}  ({len(eval_data):,} samples)")

In [ ]:
# 2b — Tokenizer + tokenized val/eval sets (for loss/perplexity)
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(str(CKPT_PATHS[0]))

def tokenize_for_loss(data_list):
    inp = tokenizer([d["input"] for d in data_list], max_length=CFG.max_input_len,
                    truncation=True, padding="max_length")
    lab = tokenizer([d["target"] for d in data_list], max_length=CFG.max_target_len,
                    truncation=True, padding="max_length")["input_ids"]
    lab = [[(t if t != tokenizer.pad_token_id else -100) for t in seq] for seq in lab]
    inp["labels"] = lab
    return inp

class LossDS(torch.utils.data.Dataset):
    def __init__(self, enc):
        self.ids = enc["input_ids"]; self.am = enc["attention_mask"]; self.lab = enc["labels"]
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        return (torch.tensor(self.ids[i]), torch.tensor(self.am[i]), torch.tensor(self.lab[i]))

val_loss_ds  = LossDS(tokenize_for_loss(val_data))
eval_loss_ds = LossDS(tokenize_for_loss(eval_data))
print(f"✅ Tokenized for loss — val: {len(val_loss_ds)}  {EVAL_NAME}: {len(eval_loss_ds)}")

In [ ]:
# 3a — Checkpoint inventory
print("=" * 74); print("CHECKPOINT INVENTORY"); print("=" * 74)
inv = []
for ck in CKPT_PATHS:
    step = int(ck.name.split("-")[1]); epoch = None
    ts = ck / "trainer_state.json"
    if ts.exists():
        epoch = json.load(open(ts)).get("epoch")
    size_mb = sum(f.stat().st_size for f in ck.rglob("*") if f.is_file()) / 1e6
    inv.append({"checkpoint": ck.name, "step": step, "epoch": epoch, "size_MB": round(size_mb, 1)})
    print(f"  📁 {ck.name:18s} | step {step:6d} | epoch {epoch} | {size_mb:8.1f} MB")
display(pd.DataFrame(inv))

In [ ]:
# 3b — Memory-safe helpers: validation loss + batched generation
from transformers import AutoModelForSeq2SeqLM
import gc

def free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_model(path):
    m = AutoModelForSeq2SeqLM.from_pretrained(str(path), torch_dtype=EVAL_DTYPE)
    return m.to(DEVICE).eval()

@torch.inference_mode()
def eval_loss(model, ds, batch_size):
    """Token-count-weighted mean cross-entropy → numerically clean perplexity."""
    tot_loss, tot_tok = 0.0, 0
    for i in range(0, len(ds), batch_size):
        ids = torch.stack([ds[j][0] for j in range(i, min(i + batch_size, len(ds)))]).to(DEVICE)
        am  = torch.stack([ds[j][1] for j in range(i, min(i + batch_size, len(ds)))]).to(DEVICE)
        lab = torch.stack([ds[j][2] for j in range(i, min(i + batch_size, len(ds)))]).to(DEVICE)
        out = model(input_ids=ids, attention_mask=am, labels=lab)
        n_tok = (lab != -100).sum().item()
        tot_loss += out.loss.item() * n_tok; tot_tok += n_tok
        del ids, am, lab, out
    free()
    return tot_loss / max(tot_tok, 1)

@torch.inference_mode()
def generate_all(model, texts, batch_size, num_beams, desc="generating"):
    preds = []
    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        chunk = texts[i:i + batch_size]
        enc = tokenizer(chunk, return_tensors="pt", padding=True, truncation=True,
                        max_length=CFG.max_input_len).to(DEVICE)
        out = model.generate(**enc, max_length=CFG.max_target_len,
                             num_beams=num_beams, early_stopping=True)
        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
        del enc, out
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return preds

print("✅ Helpers ready (fp32, batched, cache-freed).")

In [ ]:
# 3c — Select the best epoch by validation loss (the run had no eval during training)
sel = []
for ck in CKPT_PATHS:
    m = load_model(ck)
    vloss = eval_loss(m, val_loss_ds, CFG.loss_batch_size)
    sel.append({"checkpoint": ck.name, "val_loss": vloss, "val_perplexity": float(np.exp(vloss))})
    print(f"  {ck.name:18s} → val_loss {vloss:.4f}  | ppl {np.exp(vloss):.2f}")
    del m; free()

sel_df = pd.DataFrame(sel)
best_row = sel_df.loc[sel_df["val_loss"].idxmin()]
BEST_CKPT = next(p for p in CKPT_PATHS if p.name == best_row["checkpoint"])
sel_df.to_csv(FIG_DIR / "checkpoint_selection.csv", index=False)
print(f"\n🏆 Best checkpoint: {BEST_CKPT.name}  (val_loss {best_row['val_loss']:.4f})")
display(sel_df)

# Bar chart of val loss per epoch
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(sel_df["checkpoint"], sel_df["val_loss"], color=PALETTE[0], edgecolor="white")
bi = int(sel_df["val_loss"].idxmin()); bars[bi].set_color(PALETTE[2])
for b, v in zip(bars, sel_df["val_loss"]):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.3f}", ha="center", va="bottom", fontweight="bold")
ax.set_ylabel("validation loss"); ax.set_title("Checkpoint selection — validation loss by epoch")
_despine(ax); save_fig(fig, "00_checkpoint_selection")

In [ ]:
# 4a — Load the best model and generate predictions on the held-out split
model = load_model(BEST_CKPT)
print(f"Loaded {BEST_CKPT.name} in {EVAL_DTYPE} on {DEVICE}")

eval_inputs = [d["input"] for d in eval_data]
eval_refs   = [d["target"] for d in eval_data]

t0 = time.time()
eval_preds = generate_all(model, eval_inputs, CFG.gen_batch_size, CFG.gen_num_beams,
                          desc=f"generate ({EVAL_NAME})")
print(f"✅ Generated {len(eval_preds)} predictions in {(time.time()-t0)/60:.1f} min")

empty_rate = float(np.mean([len(p.strip()) == 0 for p in eval_preds]))
print(f"   Empty-generation rate: {empty_rate:.3%}")

In [ ]:
# 4b — Perplexity (held-out) via clean token-weighted loss
eval_xent = eval_loss(model, eval_loss_ds, CFG.loss_batch_size)
eval_ppl  = float(np.exp(eval_xent))
print(f"{EVAL_NAME} cross-entropy: {eval_xent:.4f}  |  perplexity: {eval_ppl:.2f}")

In [ ]:
# 5a — ROUGE (mean + per-example for distributions)
import evaluate
rouge_metric = evaluate.load("rouge")
rouge_mean = rouge_metric.compute(predictions=eval_preds, references=eval_refs, use_stemmer=True)
rouge_per  = rouge_metric.compute(predictions=eval_preds, references=eval_refs,
                                  use_stemmer=True, use_aggregator=False)
print("ROUGE:", {k: round(v, 4) for k, v in rouge_mean.items()})

In [ ]:
# 5b — BERTScore (move main model to CPU first so RoBERTa-large fits on the T4)
from bert_score import score as bert_score_fn
model.cpu(); free()
P, R, F1 = bert_score_fn(eval_preds, eval_refs, lang="en", verbose=False, device=DEVICE)
bert_p, bert_r, bert_f1 = P.mean().item(), R.mean().item(), F1.mean().item()
bert_f1_per = F1.cpu().numpy()
print(f"BERTScore — P {bert_p:.4f}  R {bert_r:.4f}  F1 {bert_f1:.4f}")
model.to(DEVICE); free()

In [ ]:
# 5c — BLEU, chrF++ (char n-gram F-score), METEOR (synonym-aware)  [NEW: chrF++, METEOR]
from sacrebleu.metrics import BLEU, CHRF
bleu_result = BLEU().corpus_score(eval_preds, [eval_refs])
chrf_result = CHRF(word_order=2).corpus_score(eval_preds, [eval_refs])   # chrF++
meteor = evaluate.load("meteor").compute(predictions=eval_preds, references=eval_refs)["meteor"]
print(f"BLEU: {bleu_result.score:.2f}  |  chrF++: {chrf_result.score:.2f}  |  METEOR: {meteor:.4f}")

In [ ]:
# 5d — Diversity: Distinct-1/2/3, intra-output repetition, Self-BLEU  [NEW: distinct-3, repetition, self-BLEU]
def _ngrams(tokens, n): return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

def distinct_n(texts, n):
    allg = []
    for t in texts: allg += _ngrams(t.lower().split(), n)
    return len(set(allg)) / len(allg) if allg else 0.0

def repetition_rate(texts, n=2):
    """Mean fraction of repeated n-grams *within* each output (degeneration signal)."""
    rates = []
    for t in texts:
        g = _ngrams(t.lower().split(), n)
        if g: rates.append(1 - len(set(g)) / len(g))
    return float(np.mean(rates)) if rates else 0.0

distinct_1, distinct_2, distinct_3 = (distinct_n(eval_preds, k) for k in (1, 2, 3))
rep_2 = repetition_rate(eval_preds, 2)

from sacrebleu import sentence_bleu
rng = random.Random(SEED)
sub_idx = rng.sample(range(len(eval_preds)), min(150, len(eval_preds)))
sub = [eval_preds[i] for i in sub_idx]
self_bleu = float(np.mean([
    sentence_bleu(sub[i], [sub[j] for j in range(len(sub)) if j != i]).score
    for i in range(len(sub))
])) if len(sub) > 1 else 0.0

print(f"Distinct-1/2/3: {distinct_1:.4f} / {distinct_2:.4f} / {distinct_3:.4f}")
print(f"Repetition (2-gram, within-output): {rep_2:.4f}   Self-BLEU(↓ better): {self_bleu:.2f}")

In [ ]:
# 5e — Abstractiveness: copy-rate & novelty vs the source input  [NEW]
# How much output text is copied from the input vs newly generated. For a
# *content specialist* that rewrites source chunks into slide bullets, some
# grounding is good, but high novelty shows it isn't merely extracting.
def copy_rate(preds, srcs, n):
    copied, total = 0, 0
    for p, s in zip(preds, srcs):
        pg = _ngrams(p.lower().split(), n); sg = set(_ngrams(s.lower().split(), n))
        if not pg: continue
        copied += sum(1 for g in pg if g in sg); total += len(pg)
    return copied / total if total else 0.0

copy_1 = copy_rate(eval_preds, eval_inputs, 1)
copy_2 = copy_rate(eval_preds, eval_inputs, 2)
novelty_2 = 1 - copy_2
print(f"Copy-rate 1-gram: {copy_1:.4f}  |  2-gram: {copy_2:.4f}  |  Novelty(2-gram): {novelty_2:.4f}")

In [ ]:
# 5f — Length stats + assemble the full metric record; save JSON + CSV
pred_lens = np.array([len(p.split()) for p in eval_preds])
ref_lens  = np.array([len(r.split()) for r in eval_refs])
in_lens   = np.array([len(s.split()) for s in eval_inputs])

METRICS = {
    "eval_split": EVAL_NAME, "n_samples": len(eval_preds), "best_checkpoint": BEST_CKPT.name,
    "rouge1": rouge_mean["rouge1"], "rouge2": rouge_mean["rouge2"], "rougeL": rouge_mean["rougeL"],
    "bertscore_p": bert_p, "bertscore_r": bert_r, "bertscore_f1": bert_f1,
    "bleu": bleu_result.score, "chrf_pp": chrf_result.score, "meteor": meteor,
    "perplexity": eval_ppl, "cross_entropy": eval_xent,
    "distinct_1": distinct_1, "distinct_2": distinct_2, "distinct_3": distinct_3,
    "repetition_2gram": rep_2, "self_bleu": self_bleu,
    "copy_rate_1gram": copy_1, "copy_rate_2gram": copy_2, "novelty_2gram": novelty_2,
    "empty_rate": empty_rate,
    "avg_pred_len": float(pred_lens.mean()), "avg_ref_len": float(ref_lens.mean()),
    "length_ratio": float(pred_lens.mean() / max(ref_lens.mean(), 1e-9)),
}
with open(FIG_DIR / "metrics.json", "w") as f:
    json.dump(METRICS, f, indent=2)
summary_df = pd.DataFrame([METRICS]).T.rename(columns={0: "value"})
summary_df.to_csv(FIG_DIR / "summary_metrics.csv")
print("💾", FIG_DIR / "metrics.json")
display(summary_df)

## 📊 Visualizations
All figures save to `/kaggle/working/figures/` and are downloadable from the
**Output** tab after a *Save & Run All* commit.

In [ ]:
# 6a — Headline metric panel (0–1 metrics) + a separate panel for the rest
norm_keys = ["rouge1", "rouge2", "rougeL", "bertscore_f1", "meteor",
             "distinct_1", "distinct_2", "distinct_3", "novelty_2gram"]
other = {"BLEU": METRICS["bleu"], "chrF++": METRICS["chrf_pp"],
         "Perplexity": METRICS["perplexity"], "Self-BLEU↓": METRICS["self_bleu"]}

fig, axes = plt.subplots(1, 2, figsize=(15, 5),
                         gridspec_kw={"width_ratios": [2.1, 1]})
vals = [METRICS[k] for k in norm_keys]
bars = axes[0].barh(range(len(norm_keys)), vals, color=PALETTE[0], edgecolor="white")
axes[0].set_yticks(range(len(norm_keys)))
axes[0].set_yticklabels([k.replace("_", " ") for k in norm_keys])
axes[0].invert_yaxis(); axes[0].set_xlim(0, 1)
for b, v in zip(bars, vals):
    axes[0].text(v + 0.01, b.get_y() + b.get_height() / 2, f"{v:.3f}", va="center", fontweight="bold")
axes[0].set_title(f"Normalized metrics (0–1) — {EVAL_NAME} set"); _despine(axes[0], left=False)

k2 = list(other.keys()); v2 = list(other.values())
b2 = axes[1].bar(k2, v2, color=PALETTE[1:5], edgecolor="white")
for b, v in zip(b2, v2):
    axes[1].text(b.get_x() + b.get_width() / 2, v, f"{v:.2f}", ha="center", va="bottom", fontweight="bold")
axes[1].set_title("Other metrics"); plt.setp(axes[1].get_xticklabels(), rotation=20, ha="right")
_despine(axes[1])
fig.suptitle(f"Content Specialist — {BEST_CKPT.name}", y=1.03)
save_fig(fig, "01_headline_metrics")

In [ ]:
# 6b — Per-example score distributions (ROUGE-L & BERTScore F1)
rougeL_per = np.array(rouge_per["rougeL"])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, name, color in [(axes[0], rougeL_per, "ROUGE-L", PALETTE[0]),
                              (axes[1], bert_f1_per, "BERTScore F1", PALETTE[2])]:
    sns.histplot(data, bins=30, kde=True, color=color, ax=ax, edgecolor="white", alpha=0.8)
    ax.axvline(np.mean(data), color=PALETTE[3], ls="--", lw=2, label=f"mean {np.mean(data):.3f}")
    ax.axvline(np.median(data), color=ACCENT, ls=":", lw=2, label=f"median {np.median(data):.3f}")
    ax.set_title(f"{name} — per-example"); ax.set_xlabel(name); ax.legend(); _despine(ax)
fig.suptitle("Score distributions across the held-out set", y=1.02)
save_fig(fig, "02_score_distributions")

In [ ]:
# 6c — Length behaviour: prediction vs reference length, and ratio
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
ax.hist(ref_lens, bins=30, alpha=0.6, color=PALETTE[0], label="reference", edgecolor="white")
ax.hist(pred_lens, bins=30, alpha=0.6, color=PALETTE[1], label="prediction", edgecolor="white")
ax.set_xlabel("length (words)"); ax.set_ylabel("count")
ax.set_title("Output length: prediction vs reference"); ax.legend(); _despine(ax)
ax = axes[1]
ax.scatter(ref_lens, pred_lens, s=14, alpha=0.4, color=PALETTE[4], edgecolor="none")
lim = max(ref_lens.max(), pred_lens.max())
ax.plot([0, lim], [0, lim], "--", color="#B9B9C6", label="y=x")
ax.set_xlabel("reference length"); ax.set_ylabel("prediction length")
ax.set_title(f"Length agreement (ratio {METRICS['length_ratio']:.2f})"); ax.legend(); _despine(ax)
save_fig(fig, "03_length_behaviour")

In [ ]:
# 6d — Does quality degrade on longer inputs? (BERTScore F1 binned by input length)
order = np.argsort(in_lens)
n_bins = 8
bins = np.array_split(order, n_bins)
xs = [in_lens[b].mean() for b in bins]
ys = [bert_f1_per[b].mean() for b in bins]
es = [bert_f1_per[b].std() for b in bins]
fig, ax = plt.subplots(figsize=(10, 5))
ax.errorbar(xs, ys, yerr=es, marker="o", ms=8, lw=2.2, color=PALETTE[0],
            ecolor="#B9B9C6", capsize=4)
ax.set_xlabel("input length (words, binned)"); ax.set_ylabel("BERTScore F1")
ax.set_title("Quality vs input length"); _despine(ax)
save_fig(fig, "04_quality_vs_input_length")

In [ ]:
# 6e — Diversity & abstractiveness summary
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
d_keys = ["distinct_1", "distinct_2", "distinct_3"]
axes[0].bar([k.replace("_", "-") for k in d_keys], [METRICS[k] for k in d_keys],
            color=PALETTE[2], edgecolor="white")
for i, k in enumerate(d_keys):
    axes[0].text(i, METRICS[k], f"{METRICS[k]:.3f}", ha="center", va="bottom", fontweight="bold")
axes[0].set_ylim(0, 1); axes[0].set_title("Lexical diversity (Distinct-n)"); _despine(axes[0])
a_keys = [("copy_rate_1gram", "copy 1g"), ("copy_rate_2gram", "copy 2g"),
          ("novelty_2gram", "novelty 2g"), ("repetition_2gram", "repetition 2g")]
axes[1].bar([lbl for _, lbl in a_keys], [METRICS[k] for k, _ in a_keys],
            color=PALETTE[1], edgecolor="white")
for i, (k, _) in enumerate(a_keys):
    axes[1].text(i, METRICS[k], f"{METRICS[k]:.3f}", ha="center", va="bottom", fontweight="bold")
axes[1].set_ylim(0, 1); axes[1].set_title("Abstractiveness vs source"); _despine(axes[1])
save_fig(fig, "05_diversity_abstractiveness")

In [ ]:
# 7a — Qualitative: best & worst 3 by BERTScore F1 + a few fixed samples
ord_f1 = np.argsort(bert_f1_per)
picks = [("WORST", ord_f1[:3]), ("BEST", ord_f1[-3:][::-1])]
rows = []
for tag, idxs in picks:
    for rank, i in enumerate(idxs, 1):
        print("=" * 80); print(f"{tag} #{rank}  ·  BERTScore F1 = {bert_f1_per[i]:.3f}")
        print("-" * 80)
        print("INPUT     :", eval_inputs[i][:200], "...")
        print("REFERENCE :", eval_refs[i][:280])
        print("GENERATED :", eval_preds[i][:280])
        rows.append({"tag": tag, "bertscore_f1": round(float(bert_f1_per[i]), 3),
                     "input": eval_inputs[i][:300], "reference": eval_refs[i][:300],
                     "generated": eval_preds[i][:300]})
pd.DataFrame(rows).to_csv(FIG_DIR / "qualitative_examples.csv", index=False)
print("\n💾", FIG_DIR / "qualitative_examples.csv")

In [ ]:
# 8a — Save the FINAL model + tokenizer + model card (session storage, no zip)
model.save_pretrained(str(CFG.out_dir))
tokenizer.save_pretrained(str(CFG.out_dir))

model_card = {
    "base_model": CFG.base_model,
    "task": "educational_slide_content_generation",
    "selected_checkpoint": BEST_CKPT.name,
    "selection_metric": "validation cross-entropy (lower is better)",
    "checkpoint_selection": sel,
    "dataset": Path(DATASET_PATH).name,
    "split": {"train": len(train_data), "val": len(val_data), "test": len(test_data)},
    "eval_split": EVAL_NAME,
    "generation": {"num_beams": CFG.gen_num_beams, "max_target_len": CFG.max_target_len},
    "final_metrics": METRICS,
    "finalized_at": datetime.datetime.now().isoformat(),
}
with open(CFG.out_dir / "model_card.json", "w") as f:
    json.dump(model_card, f, indent=2)
print(f"✅ Final model + card saved to {CFG.out_dir}")

## 📂 Outputs (session storage)
Nothing is zipped — everything below is written under `/kaggle/working/` and is
individually downloadable from the **Output** tab after a *Save & Run All* commit.

In [ ]:
# 9a — List everything saved to /kaggle/working
print("🧠 Final model →", CFG.out_dir)
for p in sorted(CFG.out_dir.rglob("*")):
    if p.is_file():
        print("   ", p.relative_to("/kaggle/working"), f"({p.stat().st_size/1e6:.1f} MB)")
print("\n🖼️  Figures & reports →", FIG_DIR)
for p in sorted(FIG_DIR.glob("*")):
    print("   ", p.relative_to("/kaggle/working"))
print("\n✅ Download any of these from the Kaggle 'Output' tab.")